# Day 6: RAG - Answering Questions From Your Own Documents

**LangChain Hands-on Series | Day 6 of 10**
Trainer: Ajeetkumar

---

Today everything clicks together. On Day 5 we built a search engine that finds the right paragraphs. On Days 2 and 3 we built prompts and chains. **RAG** (Retrieval Augmented Generation) simply puts them together:

> Before asking the LLM a question, **retrieve** the relevant paragraphs from your documents and paste them into the prompt. Then the LLM **generates** an answer from those paragraphs.

Think of an open-book exam. The student (the LLM) is smart but has never read your company handbook. RAG hands them the right pages just before they answer.

```
                           +-------------------+
   "How many sick days?" ->|     retriever     |-->  top 3 chunks from the vector store
            |              +-------------------+               |
            |                                                  v
            |            +------------------------------------------------+
            +----------->|  prompt: "Answer using ONLY this context:      |
                         |           {context}                            |
                         |           Question: {question}"                |
                         +------------------------------------------------+
                                              |
                                              v
                                            LLM  -->  "You get 12 days of paid sick leave per year."
```

Why not just fine-tune the model on your documents? RAG is cheaper, the knowledge updates the moment you add a file, and - very important in companies - you can show **which document** the answer came from.

Today's plan:

1. Rebuild the index from Day 5 (quickly)
2. Build a basic RAG chain, step by step
3. Make it say "I don't know" instead of making things up
4. Return sources and citations
5. Conversational RAG: follow-up questions like "and what about sick leave?"
6. Multi-query retrieval for better search
7. A small evaluation to check answer quality
8. **Mini project:** NovaTech HR helpdesk bot

## Setup and indexing (a quick replay of Day 5)

In [ ]:
import os
import warnings
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

warnings.filterwarnings("ignore")
load_dotenv()
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

# Load: all text/markdown policies plus the PDF
text_docs = DirectoryLoader("data/policies", glob="*.txt", loader_cls=TextLoader,
                            loader_kwargs={"encoding": "utf-8"}).load()
md_docs = TextLoader("data/policies/product_faq.md", encoding="utf-8").load()
pdf_docs = PyPDFLoader("data/policies/code_of_conduct.pdf").load()

docs = text_docs + md_docs + pdf_docs
for d in docs:
    d.metadata["file"] = os.path.basename(d.metadata["source"])      # short file name for citations

print("Loaded", len(docs), "documents")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=60).split_documents(docs)   # split
vector_store = InMemoryVectorStore.from_documents(chunks, embedding=embeddings)                     # embed + store
retriever = vector_store.as_retriever(search_kwargs={"k": 3})                                       # search interface

print("Indexed", len(chunks), "chunks")

## 1. RAG by hand, one step at a time

Before using a neat chain, let's do it manually so nothing feels like magic.

**Step 1 - Retrieve.**

In [ ]:
question = "How many days of sick leave do employees get?"

retrieved = retriever.invoke(question)

for d in retrieved:
    print(f"[{d.metadata['file']}] {d.page_content[:120]}...\n")

**Step 2 - Turn the documents into one block of text** that we can paste into a prompt. We label each piece with its source, which will help with citations later.

In [ ]:
def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata['file']}]\n{d.page_content}" for d in docs)

context = format_docs(retrieved)
print(context[:600])

**Step 3 - Write the RAG prompt.** Notice the instruction to answer **only** from the context and to say so when the answer is not there. This single sentence is your main defence against made-up answers (hallucinations).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are NovaTech's HR assistant. Answer the question using ONLY the context below. "
     "If the answer is not in the context, say: \"I could not find this in the NovaTech documents.\" "
     "Keep the answer short.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

**Step 4 - Generate.**

In [ ]:
messages = rag_prompt.invoke({"context": context, "question": question})
answer = llm.invoke(messages)

print(answer.content)

That is RAG. Four steps: retrieve, format, prompt, generate. Everything else today is about making it neater, more honest and more useful.

## 2. The same thing as one LCEL chain

Here is the classic RAG chain. Read it slowly - it uses the parallel dict and `RunnablePassthrough` from Day 3.

```
                       +--> retriever --> format_docs -->  "context"  --+
   "question" ---------+                                                +--> rag_prompt --> llm --> parser
                       +--> RunnablePassthrough() ------>  "question" --+
```

The question goes two ways at once: one copy is used for searching, the other copy goes straight into the prompt untouched.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

rag_chain.get_graph().print_ascii()

In [ ]:
print(rag_chain.invoke("How many days of sick leave do employees get?"))

In [ ]:
questions = [
    "Can I work from home during probation?",
    "What is the hotel limit in a metro city?",
    "How long is the CloudDesk free trial?",
]
for q, a in zip(questions, rag_chain.batch(questions)):
    print("Q:", q)
    print("A:", a, "\n")

## 3. Does it refuse to make things up?

**Pause and predict:** we ask something the documents never mention - "What is the CEO's salary?". What should a good RAG bot say?

In [ ]:
print(rag_chain.invoke("What is the CEO's salary?"))

<details>
<summary>Click to see the explanation</summary>

It should reply with our fallback sentence, "I could not find this in the NovaTech documents." Without that instruction, models often try to be helpful and invent a plausible number. In a company setting, "I don't know" is far better than a confident wrong answer.
</details>

Let's compare with a plain LLM that has no documents at all:

In [ ]:
print("Plain LLM:", llm.invoke("How many days of annual leave do NovaTech employees get?").content)
print()
print("RAG      :", rag_chain.invoke("How many days of annual leave do NovaTech employees get?"))

## 4. Showing sources

In real projects people will ask "where did you get that from?". Let's return the answer **and** the documents used.

The trick: retrieve the documents first and keep them, then add the answer with `RunnablePassthrough.assign`.

```
 question --> {"docs": retriever, "question": passthrough} --> assign(answer = format + prompt + llm) --> {"docs", "question", "answer"}
```

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda

answer_from_docs = (                                       # takes {"docs", "question"}, returns the answer text
    {"context": itemgetter("docs") | RunnableLambda(format_docs),
     "question": itemgetter("question")}
    | rag_prompt
    | llm
    | StrOutputParser()
)

rag_with_sources = (
    {"docs": retriever, "question": RunnablePassthrough()}
    | RunnablePassthrough.assign(answer=answer_from_docs)
)

In [ ]:
result = rag_with_sources.invoke("Is MFA mandatory for VPN?")

print("Answer:", result["answer"])
print("\nSources used:")
for d in result["docs"]:
    print(" -", d.metadata["file"], "| page", d.metadata.get("page", "-"))

### Structured answers with citations

We can go one step further with Day 2's structured output: ask the model to return the answer, the exact files it used, and how confident it is. This makes it easy to show a "Sources" section in a web app.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class CitedAnswer(BaseModel):
    answer: str = Field(description="Short answer to the question, based only on the context")
    sources: list[str] = Field(description="File names from the context that support the answer")
    confidence: Literal["high", "medium", "low"] = Field(description="How well the context supports the answer")

cited_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm.with_structured_output(CitedAnswer)
)

cited = cited_chain.invoke("What is the gift limit for vendors?")
print(cited.answer)
print("Sources   :", cited.sources)
print("Confidence:", cited.confidence)

## 5. Conversational RAG - handling follow-up questions

Real users do not ask perfect standalone questions. They chat:

> User: How many days of annual leave do I get?
> Bot: 24 days per year.
> User: **And how many can I carry forward?**

The last question makes no sense on its own - "carry forward" what? If we send it straight to the retriever, the search will be poor.

The fix is a small extra step: **rewrite the follow-up into a standalone question** using the chat history, then do normal RAG.

```
 history + "And how many can I carry forward?"
                     |
                     v
          rewrite step (LLM)  -->  "How many annual leave days can be carried forward?"
                     |
                     v
                normal RAG chain  -->  answer
```

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the chat history and the latest user question, rewrite the question so it can be "
               "understood without the history. Do NOT answer it. Return only the rewritten question."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

In [ ]:
chat_history = [
    HumanMessage(content="How many days of annual leave do I get?"),
    AIMessage(content="You get 24 days of paid annual leave per year."),
]

rewrite_chain.invoke({"chat_history": chat_history, "question": "And how many can I carry forward?"})

Now the full conversational chain. If there is no history yet, we skip the rewrite step (no point paying for an extra LLM call).

In [ ]:
def standalone_question(inputs):
    if not inputs.get("chat_history"):
        return inputs["question"]                   # first question: use as is
    return rewrite_chain.invoke(inputs)             # follow-up: rewrite it

conv_rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are NovaTech's HR assistant. Answer using ONLY the context. "
               "If it is not there, say you could not find it.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

conversational_rag = (
    RunnablePassthrough.assign(standalone=RunnableLambda(standalone_question))
    | RunnablePassthrough.assign(context=itemgetter("standalone") | retriever | format_docs)
    | conv_rag_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
history = []

def ask(question):
    answer = conversational_rag.invoke({"question": question, "chat_history": history})
    history.extend([HumanMessage(content=question), AIMessage(content=answer)])   # remember the turn
    print("You:", question)
    print("Bot:", answer, "\n")

ask("How many days of annual leave do I get?")
ask("And how many can I carry forward?")
ask("What about sick leave?")

## 6. Multi-query retrieval - asking the search engine in several ways

Sometimes the user's wording is just unlucky and the search misses the best chunk. A neat trick: ask the LLM to write **a few different versions** of the question, search with all of them, and combine the results.

```
                         +-->  "How much can I spend on a hotel?"     --> chunks A, B
 "hotel budget?" --LLM-->+-->  "What is the accommodation allowance?" --> chunks B, C
                         +-->  "Maximum hotel cost per night"         --> chunks A, D
                                                                            |
                                                         unique chunks:  A, B, C, D
```

In [ ]:
class QueryVariants(BaseModel):
    queries: list[str] = Field(description="3 different phrasings of the user's question")

variant_chain = (
    ChatPromptTemplate.from_template(
        "Write 3 different search queries that could find the answer to this question: {question}")
    | llm.with_structured_output(QueryVariants)
)

def multi_query_retrieve(question):
    variants = [question] + variant_chain.invoke({"question": question}).queries
    results = retriever.batch(variants)                      # search with every variant in parallel
    unique = {}
    for docs_for_query in results:
        for d in docs_for_query:
            unique[d.page_content] = d                       # same text = same chunk, keep it once
    return list(unique.values())

In [ ]:
q = "hotel budget?"
single = retriever.invoke(q)
multi = multi_query_retrieve(q)

print("Single query found:", len(single), "chunks")
print("Multi-query found :", len(multi), "chunks")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.bar(["single query", "multi-query"], [len(single), len(multi)], color=["#4C72B0", "#55A868"])
plt.ylabel("Unique chunks retrieved")
plt.title("Multi-query casts a wider net")
plt.show()

More chunks is not automatically better - it means more tokens in the prompt. Multi-query helps most when questions are short or vague. Use it where it improves results, not everywhere.

## 7. Is our RAG bot any good? A tiny evaluation

"It looked fine on three questions" is not testing. A simple, honest evaluation needs:

1. a small set of questions with known correct answers
2. an automatic way to grade each answer

For grading we use **LLM-as-a-judge**: another LLM call compares the bot's answer with the expected answer and returns a score. It is not perfect, but it scales far better than reading every answer by hand.

In [ ]:
test_set = [
    {"question": "How many days of annual leave per year?", "expected": "24 days"},
    {"question": "How long is paternity leave?", "expected": "15 working days"},
    {"question": "How often must passwords be changed?", "expected": "every 90 days"},
    {"question": "What is the daily food allowance for international travel?", "expected": "USD 60"},
    {"question": "How much does the Growth plan cost?", "expected": "USD 35 per agent per month"},
    {"question": "What is the gift value limit?", "expected": "below INR 2,000"},
]

class Grade(BaseModel):
    correct: bool = Field(description="True if the answer contains the same key fact as the expected answer")
    reason: str = Field(description="One short sentence explaining the grade")

judge = (
    ChatPromptTemplate.from_template(
        "Question: {question}\nExpected answer: {expected}\nBot answer: {answer}\n"
        "Does the bot answer contain the same key fact as the expected answer?")
    | llm.with_structured_output(Grade)
)

In [ ]:
import pandas as pd

answers = rag_chain.batch([t["question"] for t in test_set])
grades = judge.batch([{**t, "answer": a} for t, a in zip(test_set, answers)])

report = pd.DataFrame([
    {"question": t["question"], "expected": t["expected"], "bot_answer": a[:60], "correct": g.correct}
    for t, a, g in zip(test_set, answers, grades)
])
report

In [ ]:
accuracy = report["correct"].mean()

plt.figure(figsize=(4, 3))
plt.bar(["correct", "wrong"], [report["correct"].sum(), (~report["correct"]).sum()], color=["#55A868", "#C44E52"])
plt.title(f"RAG accuracy on the test set: {accuracy:.0%}")
plt.show()

Whenever you change something - chunk size, number of retrieved chunks `k`, the prompt - re-run this evaluation. Now you can tell whether the change actually helped instead of guessing.

## Mini project: NovaTech HR helpdesk bot

Let's package a clean, reusable bot with:
- conversational follow-ups
- structured output with sources and confidence
- a friendly printout

In [ ]:
helpdesk_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the NovaTech HR helpdesk. Use ONLY the context. Be friendly and brief. "
               "If the answer is not in the context, say so and suggest emailing hr@novatech.example."
               "\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

helpdesk_chain = (
    RunnablePassthrough.assign(standalone=RunnableLambda(standalone_question))
    | RunnablePassthrough.assign(context=itemgetter("standalone") | retriever | format_docs)
    | helpdesk_prompt
    | llm.with_structured_output(CitedAnswer)
)

class HelpdeskBot:
    def __init__(self):
        self.history = []

    def ask(self, question):
        result = helpdesk_chain.invoke({"question": question, "chat_history": self.history})
        self.history += [HumanMessage(content=question), AIMessage(content=result.answer)]
        print(f"You : {question}")
        print(f"Bot : {result.answer}")
        print(f"      sources: {', '.join(result.sources) or 'none'} | confidence: {result.confidence}\n")

In [ ]:
bot = HelpdeskBot()
bot.ask("I lost my company laptop. What should I do?")
bot.ask("Within how many hours?")
bot.ask("Do we get free lunch on Fridays?")

### Your turn

Ask the bot your own questions. Try a follow-up that only makes sense with history, and one question the documents cannot answer. Check that the sources make sense each time.

In [ ]:
my_bot = HelpdeskBot()
my_bot.ask("Can I fly business class?")

## Common mistakes on Day 6

| What you see | What usually went wrong |
|--------------|-------------------------|
| Confident answers that are not in the documents | The prompt does not say "ONLY the context" and what to do if missing |
| "I could not find this" for things that ARE in the docs | Retrieval missed it. Increase `k`, adjust chunk size, or try multi-query |
| Follow-up questions get poor answers | Add the question-rewrite step |
| `TypeError` about dict in the RAG chain | The retriever received a dict. Use `itemgetter("question")` before it |

## Quick recap

- RAG = retrieve relevant chunks, put them in the prompt, generate the answer.
- The classic chain is `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | parser`.
- Tell the model to answer only from context and to admit when it cannot.
- Return sources; use structured output for clean citations.
- Rewrite follow-up questions before retrieving.
- Measure quality with a small test set and an LLM judge.

## Practice before Day 7

1. Change `k` from 3 to 1 and to 6 and compare the evaluation accuracy.
2. Add three more questions to `test_set`, including one that should be refused.
3. Make the helpdesk bot answer in Hindi while keeping the sources in English.
4. Use `multi_query_retrieve` inside the RAG chain instead of the plain retriever.

**Tomorrow:** so far the LLM can only talk. Next we let it *act* - calling Python functions such as a calculator, a database lookup or an API.